<a href="https://colab.research.google.com/github/ajaykumar080286/GENAI_LangChain/blob/main/19_tools_in_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
!pip install langchain langchain-core langchain-community pydantic duckduckgo-search langchain_experimental ddgs

**Duck Duck Tools**

In [23]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

wrapper = DuckDuckGoSearchAPIWrapper()
search_tools = DuckDuckGoSearchRun(api_wrapper=wrapper)

In [24]:
search_tools.invoke("what is today fastival in india")

"13 hours ago · A location-aware Hindu festival calendar that combines today's Panchanga (tithi at local sunrise, nakshatra, lunar month, sankranti) with practical, cultural and family-friendly festival content. Festival dates adjust to your city. List of Indian Festivals and Holidays in the year 2026, which includes Government and National Holidays, Buddhist Holidays, Jain Holidays, Sikh Holidays and Christian Holidays in India. Indian Calendar keeps you updated with today’s date, daily Panchang, Hindu festivals, public holidays, and calendar tools — 100% accurate, free and ad-free. Holidays today, tomorrow, and upcoming holidays in India, including types like federal, national, statutory, and public holidays. Discover today's festivals, Indian and global holidays, cultural traditions, special days, festival calendars, events, and celebrations. Celebrate Every Moment with Today Festival."

**Built-in Tool - Shell Tool**

In [25]:
from langchain_community.tools import ShellTool
sheeltools=ShellTool()
sheeltools.invoke("ls -l")

Executing command:
 ls -l


'total 4\ndrwxr-xr-x 1 root root 4096 Nov 11 15:59 sample_data\n'

In [28]:
sheeltools.name
sheeltools.description
sheeltools.args

{'commands': {'anyOf': [{'type': 'string'},
   {'items': {'type': 'string'}, 'type': 'array'}],
  'description': 'List of shell commands to run. Deserialized using json.loads',
  'title': 'Commands'}}

***Custom Tools***

In [36]:
from langchain_core.tools import tool

In [37]:
#Step1- Create Funcation

In [38]:
def multiply(a,b):
  """Multiply two inputs"""
  return a*b

In [39]:
# Step 2 - Add the type hints
def multiply(a: int,b:int ) -> int :
  """Multiply two inputs"""
  return a*b


In [41]:
# Step 3 - add tool decorator
@tool
def multiply(a: int,b:int ) -> int :
  """Multiply two inputs"""
  return a*b


In [42]:
result=multiply.invoke({"a":2,"b":3})

In [48]:
result

6

In [49]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two inputs
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [52]:
multiply.args_schema.model_json_schema()

{'description': 'Multiply two inputs',
 'properties': {'a': {'title': 'A', 'type': 'integer'},
  'b': {'title': 'B', 'type': 'integer'}},
 'required': ['a', 'b'],
 'title': 'multiply',
 'type': 'object'}

***Method 2 - Using StructuredTool***

In [53]:
from langchain_core.tools import  StructuredTool
from pydantic import BaseModel,Field

In [56]:
class MultiplyInput(BaseModel):
  a: int =Field(required=True, description="First Input")
  b: int =Field(required=True, description="Second Input")

/tmp/ipykernel_403/1233915850.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  a: int =Field(required=True, description="First Input")
/tmp/ipykernel_403/1233915850.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  b: int =Field(required=True, description="Second Input")


In [57]:
def multi_func(a: int, b: int)-> int :
  """Muplti two number"""
  return a*b

In [58]:
multi_tool=StructuredTool.from_function(
    func=multi_func,
    name="multi tools",
    description="Multipky tow number",
    args_schema=MultiplyInput
)

In [59]:
result=multi_tool.invoke({"a":2,"b":2})

In [60]:
result

4

In [62]:
print(multi_tool.name)
print(multi_tool.description)
print(multi_tool.args)

multi tools
Multipky tow number
{'a': {'description': 'First Input', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'Second Input', 'title': 'B', 'type': 'integer'}}


**Method 3 - Using BaseTool Class**

In [63]:
from langchain.tools import BaseTool
from typing import Type

In [79]:
# arg schema using pydantic

class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

/tmp/ipykernel_403/908171234.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  a: int = Field(required=True, description="The first number to add")
/tmp/ipykernel_403/908171234.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  b: int = Field(required=True, description="The second number to add")


In [80]:
class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"

    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        return a * b

In [81]:
multiply_tool=MultiplyTools()

In [84]:
# Ensure name and description are strings, not tuples
multiply_tool.name = "multiply"
multiply_tool.description = "Multiply two numbers"

result = multiply_tool.invoke({'a':3, 'b':3})
print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'First Input', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'Second Input', 'title': 'B', 'type': 'integer'}}


**Toolkit**

In [98]:
from langchain_core.tools import tool


In [99]:
@tool
def addNumber(a: int, b:int)-> int:
  """Add Two Numebr """
  return a+b

In [100]:
@tool
def multiplyNumber(a: int, b:int)-> int:
  """Multiple Two Numebr """
  return a*b

In [101]:
class MathToolKit:
  def get_tools(self):
    return [addNumber, multiplyNumber]


In [102]:
toolkit=MathToolKit()

In [103]:
tools = toolkit.get_tools()


In [104]:
for tool in tools:
    print(tool.name, "=>", tool.description)


addNumber => Add Two Numebr
multiplyNumber => Multiple Two Numebr
